## Load Config 

In [1]:
import sys
from pathlib import Path

import os
from dotenv import load_dotenv
import json
sys.path.insert(0, '..')
# Load variables from .env into the environment
load_dotenv()

# Access the variables
api_key = os.getenv("OPENAI_API_KEY")
from utils.config_loader import load_config
config = load_config('../config/config.yaml')
print(config)

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from core.state import PipelineState
from agents.watchlist_agent import WatchlistContextAgent
from agents.retrieval_agent import NewsRetrievalAgent
from agents.filter_agent import NoiseFilterAgent
from agents.filter_critic_agent import FilterCriticAgent

{'retrieval': {'lookback_hours': 720, 'sources': {'sgx_announcements': True, 'newsapi': True, 'yahoo_finance': True, 'cna': True, 'straits_times': True, 'mas': True, 'sbr': True, 'reuters': True, 'nikkei_asia': True, 'cnbc_asia': True, 'reddit': False}, 'max_articles_per_query': 20, 'fetch_full_content': True}, 'newsapi_key': 'b534ae4e571a48d285f602636c3bc4e2', 'newsapi_page_size': 20, 'sgx_request_delay': 1.5, 'sgx_max_pages': 10, 'retrieval_max_workers': 8, 'filtering': {'min_snippet_length': 20, 'similarity_threshold': 0.85, 'llm_model': 'gpt-4o', 'llm_enabled': True}, 'clustering': {'method': 'agglomerative', 'min_cluster_size': 1, 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2'}, 'summarization': {'llm_model': 'gpt-4o', 'max_tokens': 512, 'temperature': 0.2, 'verification_enabled': True}, 'ranking': {'weights': {'event_type': 0.4, 'corroboration': 0.25, 'novelty': 0.2, 'credibility': 0.15}, 'source_credibility': {'tier_1': ['SGX Announcements', 'MAS'], 'tier_2': ['Busi

In [2]:
initial = PipelineState()
config['filtering']['llm_enabled'] = False
watchlist_agent = WatchlistContextAgent(config)
retrieval_agent = NewsRetrievalAgent(config)
filter_agent = NoiseFilterAgent(config)
filter_critic_agent = FilterCriticAgent(config)

## Agent Watchlist Test

In [ ]:
initial.watchlist = ["AAPL", "GOOGL", "AMZN"]
wa_agent_result = watchlist_agent.run(initial)

[2026-04-07 10:32:27] INFO WatchlistContextAgent — [WatchlistContextAgent] Starting. 3 tickers: ['AAPL', 'GOOGL', 'AMZN']
[2026-04-07 10:32:30] INFO WatchlistContextAgent — [WatchlistContextAgent] Done. 3 bundles produced


In [4]:
wa_agent_result.to_json("watchlist_test_output.json")

## Agent Watchlist Load

In [6]:
with open("watchlist_test_output.json", "r") as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
wa_agent_result = PipelineState(**data)

## Retrieval Agent

In [4]:
re_agent_result =  retrieval_agent.run(wa_agent_result) 
re_agent_result

[2026-04-07 10:43:20] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Starting. Fetching from 5 sources for 3 tickers
[2026-04-07 10:43:24] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Done. [Agent 2] ✓ 227 articles (Yahoo Finance: 49, Seeking Alpha: 16, CNBC: 6, The Motley Fool: 5, Fortune Business Insights: 5, Investing.com: 4, Reuters: 4, Search Engine Journal: 4, Search Engine Land: 4, Tom's Hardware: 3, MarketBeat: 3, Business Insider: 3, MSN: 3, Search Engine Roundtable: 3, 9to5Mac: 3, TIKR.com: 2, Barron's: 2, benzinga.com: 2, IndexBox: 2, CNN: 2, WSJ: 2, Investor's Business Daily: 2, Forbes: 2, TradingView: 2, Counterpoint Technology Market Research: 2, International Data Corporation: 2, SQ Magazine: 2, qz.com: 2, PYMNTS.com: 2, Omdia: 2, AppleInsider: 2, MacRumors: 2, Statista: 2, Straits Research: 2, Chron: 1, National Today: 1, The Guardian: 1, Stocktwits: 1, CIO Dive: 1, AASTOCKS.com: 1, The Globe and Mail: 1, MarketWatch: 1, Computerworld: 1, RVBusiness: 1, Watcher Guru: 

PipelineState(watchlist=['AAPL', 'GOOGL', 'AMZN'], query_bundles=[{'ticker': 'AAPL', 'company_name': 'Apple Inc.', 'aliases': ['Apple', 'Apple Inc'], 'industry': 'Consumer Electronics', 'company_queries': ['Apple earnings Q4 2025', 'Apple iPhone sales', 'Apple stock news'], 'industry_queries': ['smartphone market trends 2025', 'consumer electronics industry news']}, {'ticker': 'GOOGL', 'company_name': 'Alphabet Inc.', 'aliases': ['Google', 'Alphabet'], 'industry': 'Internet Services', 'company_queries': ['Google earnings Q4 2025', 'Google search engine updates', 'Alphabet stock news'], 'industry_queries': ['internet services market trends 2025', 'tech industry news']}, {'ticker': 'AMZN', 'company_name': 'Amazon.com, Inc.', 'aliases': ['Amazon', 'Amazon.com'], 'industry': 'E-commerce', 'company_queries': ['Amazon earnings Q4 2025', 'Amazon Prime subscription growth', 'Amazon stock news'], 'industry_queries': ['e-commerce market trends 2025', 'retail industry news']}], raw_articles=[{'ti

In [5]:
re_agent_result.to_json("retrieval_test_output.json")

## Retrieval Agent Load

In [3]:
with open("retrieval_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
re_agent_result = PipelineState(**data)

## News Filtering Agent

In [5]:
filter_agent_result = filter_agent.run(re_agent_result)

[2026-04-07 11:44:42] INFO NoiseFilterAgent — [NoiseFilterAgent] Starting. 227 raw articles
[2026-04-07 11:44:42] INFO NoiseFilterAgent — After hard filters: 123 articles


d:\NUS\CS5260_Project\CS5260_financial_news_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-04-07 11:44:49] INFO NoiseFilterAgent — Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3406.45it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[2026-04-07 11:44:55] INFO NoiseFilterAgent — After semantic dedup: 121 articles
[2026-04-07 11:45:07] INFO NoiseFilterAgent — After LLM filtering: 32 articles
[2026-04-07 11:45:07] INFO NoiseFilterAgent — [NoiseFilterAgent] Done. [Agent 3] ✓ 32 articles retained after all passes


In [7]:
filter_agent_result.to_json("filter_test_output.json")

## Load Filter Agent 

In [3]:
with open("filter_test_output.json", "r", encoding='utf-8') as f:
    data = json.load(f)
    
# Use the double-asterisk to unpack the dict into the class constructor
filter_agent_result = PipelineState(**data)

## Filter Critic Agent

In [4]:
filter_critic_result = filter_critic_agent.run(filter_agent_result)

[2026-04-07 13:09:22] INFO FilterCriticAgent — [FilterCriticAgent] Starting. Evaluating 32 filtered articles (attempt 2/3)
[2026-04-07 13:09:26] INFO FilterCriticAgent — [FilterCriticAgent] Done. RETRY (score 0.517 < 0.55) — 3 issue(s)


In [11]:
from langgraph.graph import StateGraph, END
from core.state import PipelineState

# Import your agent wrappers
from agents.watchlist_agent import watchlist_agent
from agents.retrieval_agent import retrieval_agent
from agents.filter_agent import filter_agent
from agents.filter_critic_agent import filter_critic_agent, route_after_filter_critic

# --- 1. Load the Config Once ---
config = load_config('../config/config.yaml')
config['filtering']['llm_enabled'] = False

def watchlist_node(state: PipelineState):
    from agents.watchlist_agent import WatchlistContextAgent
    # Combine YAML config with environment keys
    agent = WatchlistContextAgent({**config, "openai_key": os.getenv("OPENAI_API_KEY")})
    return agent.run(state)

def retrieval_node(state: PipelineState):
    from agents.retrieval_agent import NewsRetrievalAgent
    agent = NewsRetrievalAgent(config)
    return agent.run(state)

def filter_node(state: PipelineState):
    from agents.filter_agent import NoiseFilterAgent
    agent = NoiseFilterAgent(config)
    return agent.run(state)

def critic_node(state: PipelineState):
    from agents.filter_critic_agent import FilterCriticAgent
    agent = FilterCriticAgent(config.get("filtering", {}))
    return agent.run(state)

def build_financial_news_pipeline():
    # 1. Initialize the Graph with your Dataclass
    workflow = StateGraph(PipelineState)

    # 2. Add Nodes
    workflow.add_node("resolve_watchlist", watchlist_node)
    workflow.add_node("fetch_news", retrieval_node)
    workflow.add_node("filter_noise", filter_node)
    workflow.add_node("critic_eval", critic_node)
    
    # Placeholder for the next agent in your pipeline (Agent 4)
    # workflow.add_node("cluster_events", event_clustering_agent)

    # 3. Define the Flow (Edges)
    workflow.set_entry_point("resolve_watchlist")
    
    workflow.add_edge("resolve_watchlist", "fetch_news")
    workflow.add_edge("fetch_news", "filter_noise")
    workflow.add_edge("filter_noise", "critic_eval")

    # 4. Add Conditional Routing (The Loop)
    workflow.add_conditional_edges(
        "critic_eval",
        route_after_filter_critic,
        {
            "retry_filter": "filter_noise",  # Loop back to Agent 3
            "continue": END,                # Or link to "cluster_events"
            "abort": END                    # End on critical error
        }
    )

    return workflow.compile()


In [15]:
app = build_financial_news_pipeline()
    
initial_state = PipelineState(watchlist=["TSLA", "NVDA", "AAPL"])
final_state = app.invoke(initial_state)
    
print(f"Final Article Count: {final_state['clean_article_count']}")
print(f"Retries attempted: {final_state['filter_retry_count']}")

[2026-04-07 13:40:05] INFO WatchlistContextAgent — [WatchlistContextAgent] Starting. 3 tickers: ['TSLA', 'NVDA', 'AAPL']
[2026-04-07 13:40:09] INFO WatchlistContextAgent — [WatchlistContextAgent] Done. 3 bundles produced
[2026-04-07 13:40:09] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Starting. Fetching from 5 sources for 3 tickers
[2026-04-07 13:40:13] INFO NewsRetrievalAgent — [NewsRetrievalAgent] Done. [Agent 2] ✓ 231 articles (Yahoo Finance: 55, Seeking Alpha: 12, fool.com: 7, CNBC: 7, Investing.com: 5, TipRanks: 4, Benzinga: 4, Tom's Hardware: 4, Reuters: 4, PR Newswire: 3, Quiver Quantitative: 3, Investor's Business Daily: 3, omdia.tech.informa.com: 3, 36氪: 2, Autonocion.com: 2, TradingView: 2, Barron's: 2, thestreet.com: 2, MarketBeat: 2, IndexBox: 2, Autoblog: 2, Semiconductor Industry Association | SIA: 2, TechCrunch: 2, Counterpoint Technology Market Research: 2, 9to5Mac: 2, nvidianews.nvidia.com: 2, AppleInsider: 2, EVTech.News: 1, The Korea Times: 1, The Driven: 1, Tesl

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4934.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[2026-04-07 13:40:18] INFO NoiseFilterAgent — After semantic dedup: 130 articles
[2026-04-07 13:40:18] INFO NoiseFilterAgent — [NoiseFilterAgent] Done. [Agent 3] ✓ 130 articles retained after all passes
[2026-04-07 13:40:18] INFO FilterCriticAgent — [FilterCriticAgent] Starting. Evaluating 130 filtered articles (attempt 2/3)
[2026-04-07 13:40:22] INFO FilterCriticAgent — [FilterCriticAgent] Done. PASS (score 0.586 ≥ 0.55)
Final Article Count: 130
Retries attempted: 1


In [14]:
final_state['clean_article_count']

136